# Task 2: Data Engineering and Preparation

## 1. Import libraries

In [121]:
import pandas as pd
import numpy as np
from IPython.display import display
from datetime import datetime

## 2. Helper Functions

These helper functions are used throughout the notebook to:
- inspect data quality
- standardize column names
- convert raw missing markers into proper missing values
- safely divide ratios
- fill numeric columns with median


In [ ]:
# ============================================================
# HELPER FUNCTIONS FOR DATA QUALITY & CLEANING
# ============================================================
# Rationale: Modularizing data cleaning steps into functions ensures reproducibility 
# and prevents code duplication across multiple datasets.

# Displays a quick summary of dataset quality including shape, 
# duplicates, and missing values to guide cleaning decisions.
def quality_report(df, name):
    print("\n" + "=" * 70)
    print(f"DATA QUALITY REPORT: {name}")
    print("=" * 70)
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values per column:")
    print(df.isnull().sum().sort_values(ascending=False))


# Standardizes column names for consistency
# Converts to lowercase, replaces spaces with underscores, and removes special characters
def standardize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace(r"[^\w\s]", "", regex=True)
    )
    return df


# Converts raw missing markers into proper NaN values
# Crucial for accurate statistical computing and imputation
def normalize_missing_markers(df):
    df = df.copy()
    df = df.replace(["-", " - ", "", " ", "NA", "N/A", "na", "n/a", "null", "Null"], np.nan)
    return df


# Safe division for ratios, preventing Infinity or ZeroDivisionError
# Important when calculating gap metrics where the denominator might be 0
def safe_divide(numerator, denominator):
    result = numerator / denominator.replace(0, np.nan)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result


# Evaluates the effectiveness of missing value handling steps
def missing_comparison(before_df, after_df, title):
    before_missing = before_df.isnull().sum()
    after_missing = after_df.isnull().sum()

    comparison = pd.DataFrame({
        "Before Cleaning": before_missing,
        "After Cleaning": after_missing,
        "Reduction": before_missing - after_missing
    }).sort_values("Before Cleaning", ascending=False)

    print(f"\n{'='*70}")
    print(f"MISSING VALUE COMPARISON: {title}")
    print(f"{'='*70}")
    display(comparison)
    return comparison


# Two-tier Imputation Strategy
# Rationale: Filling missing values with a global median could distort country-specific trends.
# We first impute with the group median (e.g., country-year) to preserve local distribution,
# and only fall back to the global median if the entire group is NaN.
def fill_by_group_then_global(df, group_cols, numeric_cols):
    df = df.copy()

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

        # Group-level median fill
        df[col] = df.groupby(group_cols)[col].transform(
            lambda x: x.fillna(x.median())
        )

        # Global median fallback for completely missing groups
        df[col] = df[col].fillna(df[col].median())

    return df


## 3. Load both datasets

In [ ]:
pip_df = pd.read_csv("World Bank Poverty & Inequality.csv")
wdi_df = pd.read_csv("World Development Indicators.csv")


## 4. Initial Data Quality Assessment

In [124]:
quality_report(pip_df, "Raw PIP Dataset")
quality_report(wdi_df, "Raw WDI Dataset")


DATA QUALITY REPORT: Raw PIP Dataset
Shape: (255600, 10)
Duplicate rows: 0

Missing values per column:
quantile           2556
country_code          0
year                  0
reporting_level       0
percentile            0
welfare_type          0
avg_welfare           0
pop_share             0
welfare_share         0
pop                   0
dtype: int64

DATA QUALITY REPORT: Raw WDI Dataset
Shape: (2665, 10)
Duplicate rows: 2

Missing values per column:
Time Code                                                                               5
Country Name                                                                            5
School enrollment, secondary (% gross) [SE.SEC.ENRR]                                    5
Country Code                                                                            5
GDP per capita (current US$) [NY.GDP.PCAP.CD]                                           5
Life expectancy at birth, total (years) [SP.DYN.LE00.IN]                                5
A

In [125]:
print("PIP preview:")
display(pip_df.head())

print("WDI preview:")
display(wdi_df.head())

PIP preview:


,country_code,year,reporting_level,welfare_type,percentile,avg_welfare,pop_share,welfare_share,quantile,pop
0,AGO,2000,national,consumption,1,0.285539,0.01,0.000341,0.312285,54610.07542
1,AGO,2000,national,consumption,2,0.405434,0.01,0.000484,0.472927,54610.07542
2,AGO,2000,national,consumption,3,0.554057,0.01,0.000661,0.628385,54610.07542
3,AGO,2000,national,consumption,4,0.692758,0.01,0.000827,0.761937,54610.07542
4,AGO,2000,national,consumption,5,0.809901,0.01,0.000966,0.849781,54610.07542


WDI preview:


,Time,Time Code,Country Name,Country Code,GDP per capita (current US$) [NY.GDP.PCAP.CD],"Life expectancy at birth, total (years) [SP.DYN.LE00.IN]","School enrollment, secondary (% gross) [SE.SEC.ENRR]","Population, total [SP.POP.TOTL]",Access to electricity (% of population) [EG.ELC.ACCS.ZS],"Unemployment, total (% of total labor force) (modeled ILO estimate) [SL.UEM.TOTL.ZS]"
0,2016,YR2016,Afghanistan,AFG,522.082215583898,62.646,53.4350395202637,34700612,97.7,10.116
1,2016,YR2016,Albania,ALB,4457.63412207495,78.643,99.170755047578,2689469,99.9,15.418
2,2016,YR2016,Algeria,DZA,4424.98529027556,75.31,..,40850721,99.4,10.202
3,2016,YR2016,American Samoa,ASM,12843.3342903627,72.64,..,52245,..,..
4,2016,YR2016,Andorra,AND,40129.8385811474,84.489,94.6524124145508,72181,100,..


## 5. Clean the Poverty & Inequality Dataset (PIP)

This section:
- standardizes column names
- converts raw missing markers to NaN
- converts important columns to numeric
- fills missing numeric values using median
- filters the dataset to valid national-level recent observations
- removes impossible values

In [ ]:
# ============================================================
# PIP DATASET CLEANING & VALIDATION
# ============================================================
# Rationale: Standardize the Poverty and Inequality (PIP) dataset 
# by aligning columns, removing invalid data, and imputing missing fields 
# to calculate accurate distribution metrics downstream.

def clean_pip_data(pip_df):
    pip_df = pip_df.copy()
    pip_before = pip_df.copy()

    # Apply standardization and NaN conversions
    pip_df = standardize_columns(pip_df)
    pip_df = normalize_missing_markers(pip_df)
    pip_df = pip_df.drop_duplicates()

    # Specify columns that are strictly numeric and require imputation
    numeric_cols = [
        "percentile",
        "avg_welfare",
        "pop_share",
        "welfare_share",
        "quantile",
        "pop"
    ]

    # Convert year separately (we never median-fill year, as it's a temporal identifier)
    pip_df["year"] = pd.to_numeric(pip_df["year"], errors="coerce")

    # Drop rows missing absolute key identifiers (can't aggregate without these)
    pip_df = pip_df.dropna(subset=["country_code", "year", "reporting_level", "welfare_type"])

    # Decision: Keep only 'national' reporting to ensure comparability
    # Decision: Time period 2019-2024 focuses on recent patterns (including post-pandemic)
    pip_df = pip_df[
        (pip_df["reporting_level"] == "national") &
        (pip_df["welfare_type"].isin(["income", "consumption"])) &
        (pip_df["year"].between(2019, 2024))
    ].copy()

    # Decision: Impute missing numerical values using country-year medians first.
    # We prefer the median over the mean because wealth/welfare distributions 
    # are heavily right-skewed and sensitive to outliers.
    pip_df = fill_by_group_then_global(
        pip_df,
        group_cols=["country_code", "year"],
        numeric_cols=numeric_cols
    )

    # Sanity checks: Remove mathematically impossible values that indicate data errors
    pip_df = pip_df[
        pip_df["percentile"].between(1, 100) &
        pip_df["pop_share"].between(0, 1) &
        pip_df["welfare_share"].between(0, 1) &
        (pip_df["avg_welfare"] >= 0) &
        (pip_df["pop"] >= 0)
    ].copy()

    # Track partial years (e.g. 2024) to adjust interpretations during EDA
    pip_df["is_partial_year"] = pip_df["year"].eq(2024)

    # Evaluate cleaning strategy outcomes
    missing_comparison(
        normalize_missing_markers(standardize_columns(pip_before)),
        pip_df,
        "PIP Dataset"
    )

    return pip_df


pip_clean = clean_pip_data(pip_df)

quality_report(pip_clean, "Cleaned PIP Dataset")
display(pip_clean.head())



MISSING VALUE COMPARISON: PIP Dataset


,Before Cleaning,After Cleaning,Reduction
quantile,2556.0,0,2556.0
avg_welfare,0.0,0,0.0
country_code,0.0,0,0.0
percentile,0.0,0,0.0
pop,0.0,0,0.0
pop_share,0.0,0,0.0
reporting_level,0.0,0,0.0
welfare_share,0.0,0,0.0
welfare_type,0.0,0,0.0
year,0.0,0,0.0



DATA QUALITY REPORT: Cleaned PIP Dataset
Shape: (36800, 11)
Duplicate rows: 0

Missing values per column:
country_code       0
year               0
reporting_level    0
welfare_type       0
percentile         0
avg_welfare        0
pop_share          0
welfare_share      0
quantile           0
pop                0
is_partial_year    0
dtype: int64


,country_code,year,reporting_level,welfare_type,percentile,avg_welfare,pop_share,welfare_share,quantile,pop,is_partial_year
1600,ALB,2019,national,consumption,1,3.257677,0.01,0.002226,3.791016,28472.751151,False
1601,ALB,2019,national,consumption,2,4.037495,0.01,0.002759,4.326269,28472.751151,False
1602,ALB,2019,national,consumption,3,4.573097,0.01,0.003125,4.777535,28472.751151,False
1603,ALB,2019,national,consumption,4,4.901667,0.01,0.003350,5.029792,28472.751151,False
1604,ALB,2019,national,consumption,5,5.174110,0.01,0.003536,5.337046,28472.751151,False


## 6. Validate the Cleaned PIP Dataset

In [127]:
# Check important PIP columns for remaining missing values
pip_key_columns = [
    "country_code",
    "year",
    "percentile",
    "avg_welfare",
    "pop_share",
    "welfare_share",
    "quantile",
    "pop"
]

print("Missing values in key PIP columns:")
print(pip_clean[pip_key_columns].isnull().sum())

Missing values in key PIP columns:
country_code     0
year             0
percentile       0
avg_welfare      0
pop_share        0
welfare_share    0
quantile         0
pop              0
dtype: int64


## 7. Aggregate PIP to Country-Year Level

The PIP dataset is originally reported at percentile level. To merge it with WDI indicators, it must be aggregated to country-year level.

This section also engineers advanced inequality features such as:
- top 20 vs bottom 20 welfare ratio
- p90 vs p10 welfare ratio
- welfare gap
- welfare dispersion measures

In [ ]:
# ============================================================
# WELFARE INDEX AGGREGATION & FEATURE ENGINEERING
# ============================================================
# Rationale: The raw dataset contains up to 100 percentile bins per country-year.
# We map these bins into meaningful inequality metrics at a broader level:
# 1) Reduce dimensional noise while retaining distributional shape.
# 2) Emphasize Top 20% vs Bottom 20% to directly address inequality dynamics.
# 3) Provide single country-year values suitable for merging with WDI data.

def aggregate_pip_data(pip_df):
    pip_df = pip_df.copy()

    # Step 1: Base summary metrics per country-year (mean, median, dispersion)
    base = pip_df.groupby(["country_code", "year"]).agg(
        mean_avg_welfare=("avg_welfare", "mean"),
        median_quantile=("quantile", "median"),
        welfare_std=("avg_welfare", "std"),      # Variance of welfare distribution
        welfare_iqr=("avg_welfare", lambda x: x.quantile(0.75) - x.quantile(0.25)),
        total_pop=("pop", "max"),
        is_partial_year=("is_partial_year", "max")
    ).reset_index()

    # Step 2: Compute share metrics across different percentiles
    # This addresses wealth distribution across socio-economic tiers.
    bottom20 = pip_df[pip_df["percentile"] <= 20].groupby(["country_code", "year"]).agg(
        bottom20_avg_welfare=("avg_welfare", "mean"),
        bottom20_welfare_share=("welfare_share", "sum")
    ).reset_index()

    bottom40 = pip_df[pip_df["percentile"] <= 40].groupby(["country_code", "year"]).agg(
        bottom40_welfare_share=("welfare_share", "sum")
    ).reset_index()

    top20 = pip_df[pip_df["percentile"] > 80].groupby(["country_code", "year"]).agg(
        top20_avg_welfare=("avg_welfare", "mean"),
        top20_welfare_share=("welfare_share", "sum")
    ).reset_index()

    top10 = pip_df[pip_df["percentile"] > 90].groupby(["country_code", "year"]).agg(
        top10_avg_welfare=("avg_welfare", "mean"),
        top10_welfare_share=("welfare_share", "sum")
    ).reset_index()

    # Extreme percentiles to identify massive gaps (P90-P10 gap)
    p10 = pip_df[pip_df["percentile"] <= 10].groupby(["country_code", "year"]).agg(
        p10_avg_welfare=("avg_welfare", "mean")
    ).reset_index()

    p90 = pip_df[pip_df["percentile"] > 90].groupby(["country_code", "year"]).agg(
        p90_avg_welfare=("avg_welfare", "mean")
    ).reset_index()

    # Consolidate all groupings
    result = (
        base.merge(bottom20, on=["country_code", "year"], how="left")
            .merge(bottom40, on=["country_code", "year"], how="left")
            .merge(top20, on=["country_code", "year"], how="left")
            .merge(top10, on=["country_code", "year"], how="left")
            .merge(p10, on=["country_code", "year"], how="left")
            .merge(p90, on=["country_code", "year"], how="left")
    )

    # Step 3: Compute advanced inequality ratios using safe_divide (prevents /0)
    # The Top 20:Bottom 20 ratio describes general inequality
    result["top20_bottom20_ratio"] = safe_divide(
        result["top20_avg_welfare"],
        result["bottom20_avg_welfare"]
    )

    # The 90:10 ratio describes extreme disparity 
    result["p90_p10_ratio"] = safe_divide(
        result["p90_avg_welfare"],
        result["p10_avg_welfare"]
    )

    # The welfare gap shows raw percentage difference in share
    result["welfare_gap"] = result["top20_welfare_share"] - result["bottom20_welfare_share"]

    # Step 4: Final Imputation of Engineered Metrics
    # Since combining multiple groupings might result in nulls (e.g., if a country 
    # lacks P90 observations but has P10 observations), we ensure numerical stability 
    # by filling residual NaNs with medians.
    numeric_fill_cols = [
        "welfare_std",
        "welfare_iqr",
        "bottom20_avg_welfare",
        "bottom20_welfare_share",
        "bottom40_welfare_share",
        "top20_avg_welfare",
        "top20_welfare_share",
        "top10_avg_welfare",
        "top10_welfare_share",
        "p10_avg_welfare",
        "p90_avg_welfare",
        "top20_bottom20_ratio",
        "p90_p10_ratio",
        "welfare_gap"
    ]

    for col in numeric_fill_cols:
        result[col] = pd.to_numeric(result[col], errors="coerce")
        result[col] = result[col].fillna(result[col].median())

    return result


pip_country_year = aggregate_pip_data(pip_clean)

quality_report(pip_country_year, "Aggregated PIP Country-Year Dataset")
display(pip_country_year.head())



DATA QUALITY REPORT: Aggregated PIP Country-Year Dataset
Shape: (358, 20)
Duplicate rows: 0

Missing values per column:
country_code              0
year                      0
mean_avg_welfare          0
median_quantile           0
welfare_std               0
welfare_iqr               0
total_pop                 0
is_partial_year           0
bottom20_avg_welfare      0
bottom20_welfare_share    0
bottom40_welfare_share    0
top20_avg_welfare         0
top20_welfare_share       0
top10_avg_welfare         0
top10_welfare_share       0
p10_avg_welfare           0
p90_avg_welfare           0
top20_bottom20_ratio      0
p90_p10_ratio             0
welfare_gap               0
dtype: int64


,country_code,year,mean_avg_welfare,median_quantile,welfare_std,welfare_iqr,total_pop,is_partial_year,bottom20_avg_welfare,bottom20_welfare_share,bottom40_welfare_share,top20_avg_welfare,top20_welfare_share,top10_avg_welfare,top10_welfare_share,p10_avg_welfare,p90_avg_welfare,top20_bottom20_ratio,p90_p10_ratio,welfare_gap
0,ALB,2019,14.631818,12.356349,8.727751,9.127375,28472.751151,False,6.225324,0.085093,0.214681,28.428645,0.388587,34.582304,0.236350,5.170503,34.582304,4.566613,6.688383,0.303494
1,ALB,2020,14.847158,12.838601,8.448225,9.211061,28031.916239,False,6.267664,0.084429,0.216421,28.227366,0.380239,33.791613,0.227597,5.067823,33.791613,4.503650,6.667876,0.295810
2,ARM,2019,9.087722,7.717232,6.495384,4.861010,28849.577912,False,4.060171,0.089355,0.221330,17.817061,0.392113,22.943118,0.252463,3.406026,22.943118,4.388254,6.736037,0.302758
3,ARM,2020,8.589780,7.608539,4.513978,4.245442,27146.378537,False,4.392951,0.102283,0.245344,15.258835,0.355279,18.430578,0.214564,3.759291,18.430578,3.473481,4.902674,0.252996
4,ARM,2021,8.941558,7.749302,5.566521,4.569458,26523.512035,False,4.210894,0.094187,0.231773,16.851626,0.376928,21.059979,0.235529,3.545769,21.059979,4.001912,5.939467,0.282741


## 8. Clean the World Development Indicators (WDI) Dataset

This section:
- renames long indicator columns
- handles raw missing markers such as "-"
- converts important fields to numeric
- fills numeric missing values using median
- filters to recent years only

In [ ]:
# ============================================================
# WDI DATASET CLEANING & VALIDATION
# ============================================================
# Rationale: The World Development Indicators (WDI) dataset uses verbose column titles 
# and varied missing data indicators (e.g., "-"). We must standardize structural keys 
# (Country Code, Year) to ensure a perfect 1:1 join with the PIP dataset.

def clean_wdi_data(wdi_df):
    # Create working copies for auditing
    wdi_df = wdi_df.copy()
    wdi_before = wdi_df.copy()

    # Step 1: Normalize raw missing markers
    # Certain WDI extracts represent missing records with symbols instead of NaNs,
    # which disrupts numeric conversions.
    wdi_df = wdi_df.replace([
        "-", " - ", "", " ", "NA", "N/A", "na", "n/a", "null", "Null"
    ], np.nan)

    # Step 2: Remove strict duplicates
    wdi_df = wdi_df.drop_duplicates()

    # Step 3: Rename verbose column headers for clean downstream coding
    wdi_df = wdi_df.rename(columns={
        "Time": "year",
        "Time Code": "time_code",
        "Country Name": "country_name",
        "Country Code": "country_code",
        "GDP per capita (current US$) [NY.GDP.PCAP.CD]": "gdp_per_capita",
        "Life expectancy at birth, total (years) [SP.DYN.LE00.IN]": "life_expectancy",
        "School enrollment, secondary (% gross) [SE.SEC.ENRR]": "school_enrollment_secondary",
        "Population, total [SP.POP.TOTL]": "population_total",
        "Access to electricity (% of population) [EG.ELC.ACCS.ZS]": "electricity_access",
        "Unemployment, total (% of total labor force) (modeled ILO estimate) [SL.UEM.TOTL.ZS]": "unemployment_total"
    })
    
    # Step 4: Temporal Alignment (Convert YEAR strictly)
    # Note: We avoid median filling on `year` as temporal data cannot simply be interpolated globally
    wdi_df["year"] = pd.to_numeric(wdi_df["year"], errors="coerce")

    # Drop core unidentifiable rows
    wdi_df = wdi_df.dropna(subset=["country_code", "country_name", "year"])

    # Align temporal window to 2019-2024 to match our PIP target
    wdi_df = wdi_df[wdi_df["year"].between(2019, 2024)].copy()

    # Step 5: Isolate relevant socio-economic descriptors
    numeric_cols = [
        "gdp_per_capita",
        "life_expectancy",
        "school_enrollment_secondary",
        "population_total",
        "electricity_access",
        "unemployment_total"
    ]

    # Step 6: SAFE GROUP-BASED IMPUTATION 
    # Macroeconomic stats shift differently per country. Imputing a missed "Life Expectancy"
    # for a developing nation using a developed nation's median would skew reality. 
    # Therefore, we impute relative to the Country's specific historical median.
    def safe_group_fill(series):
        # Fallback safeguard if all records for the country are NaN
        if series.notna().sum() == 0:
            return series
        else:
            return series.fillna(series.median())

    for col in numeric_cols:
        wdi_df[col] = pd.to_numeric(wdi_df[col], errors="coerce")

        # Localised country-level median fill
        wdi_df[col] = wdi_df.groupby("country_code")[col].transform(safe_group_fill)

        # Global median fallback (if even after localized fill we have gaps)
        if wdi_df[col].notna().sum() > 0:
            wdi_df[col] = wdi_df[col].fillna(wdi_df[col].median())
        else:
            wdi_df[col] = wdi_df[col].fillna(0)

   
    # Step 7: Evaluate Missing Data Handling Outcomes
    before = wdi_before.replace([
        "-", " - ", "", " ", "NA", "N/A", "na", "n/a", "null", "Null"
    ], np.nan)

    before = before.rename(columns={
        "Time": "year",
        "Time Code": "time_code",
        "Country Name": "country_name",
        "Country Code": "country_code",
        "GDP per capita (current US$) [NY.GDP.PCAP.CD]": "gdp_per_capita",
        "Life expectancy at birth, total (years) [SP.DYN.LE00.IN]": "life_expectancy",
        "School enrollment, secondary (% gross) [SE.SEC.ENRR]": "school_enrollment_secondary",
        "Population, total [SP.POP.TOTL]": "population_total",
        "Access to electricity (% of population) [EG.ELC.ACCS.ZS]": "electricity_access",
        "Unemployment, total (% of total labor force) (modeled ILO estimate) [SL.UEM.TOTL.ZS]": "unemployment_total"
    })

    before_missing = before.isnull().sum()
    after_missing = wdi_df.isnull().sum()

    comparison = pd.DataFrame({
        "Before Cleaning": before_missing,
        "After Cleaning": after_missing,
        "Reduction": before_missing - after_missing
    }).sort_values("Before Cleaning", ascending=False)

    print("\n" + "=" * 70)
    print("MISSING VALUE COMPARISON: WDI DATASET")
    print("=" * 70)
    display(comparison)

    return wdi_df

wdi_clean = clean_wdi_data(wdi_df)

# Final structural verification
print("\nMissing values after full WDI cleaning:")
display(wdi_clean.isnull().sum())

print("\nPreview of cleaned WDI dataset:")
display(wdi_clean.head())


Missing values after full WDI cleaning:


year                           0
time_code                      0
country_name                   0
country_code                   0
gdp_per_capita                 0
life_expectancy                0
school_enrollment_secondary    0
population_total               0
electricity_access             0
unemployment_total             0
dtype: int64


Preview of cleaned WDI dataset:


,year,time_code,country_name,country_code,gdp_per_capita,life_expectancy,school_enrollment_secondary,population_total,electricity_access,unemployment_total
798,2019.0,YR2019,Afghanistan,AFG,496.602504,62.941,91.235570,37856121.0,97.7,11.1870
799,2019.0,YR2019,Albania,ALB,6069.439031,79.467,99.015154,2567801.0,100.0,11.4660
800,2019.0,YR2019,Algeria,DZA,4468.453419,75.682,104.550270,43294546.0,99.5,12.3020
801,2019.0,YR2019,American Samoa,ASM,12886.135952,72.751,91.235570,50209.0,99.9,5.6215
802,2019.0,YR2019,Andorra,AND,41257.816459,84.098,96.574333,76474.0,100.0,5.6215


## 10. Validate the Cleaned WDI Dataset

In [130]:
wdi_key_columns = [
    "country_code",
    "country_name",
    "year",
    "gdp_per_capita",
    "life_expectancy",
    "school_enrollment_secondary",
    "population_total",
    "electricity_access",
    "unemployment_total"
]

print("Missing values in key WDI columns:")
print(wdi_clean[wdi_key_columns].isnull().sum())

Missing values in key WDI columns:
country_code                   0
country_name                   0
year                           0
gdp_per_capita                 0
life_expectancy                0
school_enrollment_secondary    0
population_total               0
electricity_access             0
unemployment_total             0
dtype: int64


## 11. Merge the Cleaned PIP and WDI Datasets

The cleaned country-year PIP dataset is merged with the cleaned WDI dataset
using country code and year as the common keys.

In [131]:
merged_df = pd.merge(
    pip_country_year,
    wdi_clean,
    on=["country_code", "year"],
    how="inner"
)

quality_report(merged_df, "Merged Dataset Before Feature Engineering")
print("\nMerged Dataset Preview:")
display(merged_df.head())


DATA QUALITY REPORT: Merged Dataset Before Feature Engineering
Shape: (355, 28)
Duplicate rows: 0

Missing values per column:
country_code                   0
year                           0
mean_avg_welfare               0
median_quantile                0
welfare_std                    0
welfare_iqr                    0
total_pop                      0
is_partial_year                0
bottom20_avg_welfare           0
bottom20_welfare_share         0
bottom40_welfare_share         0
top20_avg_welfare              0
top20_welfare_share            0
top10_avg_welfare              0
top10_welfare_share            0
p10_avg_welfare                0
p90_avg_welfare                0
top20_bottom20_ratio           0
p90_p10_ratio                  0
welfare_gap                    0
time_code                      0
country_name                   0
gdp_per_capita                 0
life_expectancy                0
school_enrollment_secondary    0
population_total               0
electricity_acc

,country_code,year,mean_avg_welfare,median_quantile,welfare_std,welfare_iqr,total_pop,is_partial_year,bottom20_avg_welfare,bottom20_welfare_share,...,p90_p10_ratio,welfare_gap,time_code,country_name,gdp_per_capita,life_expectancy,school_enrollment_secondary,population_total,electricity_access,unemployment_total
0,ALB,2019,14.631818,12.356349,8.727751,9.127375,28472.751151,False,6.225324,0.085093,...,6.688383,0.303494,YR2019,Albania,6069.439031,79.467000,99.015154,2567801.0,100.0,11.466
1,ALB,2020,14.847158,12.838601,8.448225,9.211061,28031.916239,False,6.267664,0.084429,...,6.667876,0.295810,YR2020,Albania,6027.913507,77.824000,95.929447,2528480.0,100.0,11.639
2,ARM,2019,9.087722,7.717232,6.495384,4.861010,28849.577912,False,4.060171,0.089355,...,6.736037,0.302758,YR2019,Armenia,4597.228874,76.221951,86.826063,2962500.0,100.0,18.304
3,ARM,2020,8.589780,7.608539,4.513978,4.245442,27146.378537,False,4.392951,0.102283,...,4.902674,0.252996,YR2020,Armenia,4268.680933,73.375610,88.508030,2961500.0,100.0,18.175
4,ARM,2021,8.941558,7.749302,5.566521,4.569458,26523.512035,False,4.210894,0.094187,...,5.939467,0.282741,YR2021,Armenia,4685.179971,72.278049,91.485374,2962300.0,100.0,15.469


## 12. Advanced Feature Engineering

This section creates:
- log GDP per capita
- datetime column
- decade grouping
- inequality category
- normalized development indicators
- composite development score

In [ ]:
# ============================================================
# ADVANCED MULTIDIMENSIONAL FEATURE ENGINEERING
# ============================================================
# Rationale: Raw metric formats do not immediately reveal complex patterns of poverty.
# We transform relationships here (e.g. logging GDP to stabilize right-skew bias) and 
# normalize indicators into a generic composite score ranging from 0-1, so that units 
# like "years" (life expectancy) can seamlessly mix with "dollars" (GDP).

def engineer_features(df):
    df = df.copy()

    # Log Transformation: GDP distributions are exponentially right-skewed.
    # Applying `log1p` corrects heteroscedasticity globally, making variations clearer.
    df["log_gdp_per_capita"] = np.log1p(df["gdp_per_capita"])

    # Temporality tracking for continuous timeline plotting
    df["date"] = pd.to_datetime(df["year"].astype(int).astype(str) + "-01-01")
    df["decade"] = (df["year"] // 10) * 10

    # Categorical Bucket creation explicitly for categorical visual encodings
    # Dividing purely numeric ratios into High, Medium, Low buckets via qcut
    df["inequality_category"] = pd.qcut(
        df["top20_bottom20_ratio"],
        q=3,
        labels=["Low inequality", "Medium inequality", "High inequality"],
        duplicates="drop"
    )

    # Indicator Normalization using Min-Max Scaling [0, 1] Range.
    # Assumption: Bounding these 4 features onto the same scale allows 
    # their geometric average to act as an unweighted Multi-Dimensional Development measure.
    score_cols = [
        "gdp_per_capita",
        "life_expectancy",
        "school_enrollment_secondary",
        "electricity_access"
    ]

    for col in score_cols:
        # Min-Max Scaling formulation
        df[col + "_norm"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

    # Creating the Composite Structural Development Metric
    # Averaging the normalized vectors produces an interpretable structural strength index
    df["development_score"] = df[[col + "_norm" for col in score_cols]].mean(axis=1)

    # Classifying the derived Development Multi-Index
    df["development_tier"] = pd.qcut(
        df["development_score"],
        q=3,
        labels=["Low development", "Medium development", "High development"],
        duplicates="drop"
    )

    # Ensuring stability by guaranteeing numeric integrity post-engineering
    feature_numeric_cols = [
        "log_gdp_per_capita",
        "gdp_per_capita_norm",
        "life_expectancy_norm",
        "school_enrollment_secondary_norm",
        "electricity_access_norm",
        "development_score"
    ]

    for col in feature_numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df[col] = df[col].fillna(df[col].median())

    return df


merged_df = engineer_features(merged_df)

quality_report(merged_df, "Merged Dataset After Feature Engineering")
display(merged_df.head())


DATA QUALITY REPORT: Merged Dataset After Feature Engineering
Shape: (355, 38)
Duplicate rows: 0

Missing values per column:
country_code                        0
year                                0
mean_avg_welfare                    0
median_quantile                     0
welfare_std                         0
welfare_iqr                         0
total_pop                           0
is_partial_year                     0
bottom20_avg_welfare                0
bottom20_welfare_share              0
bottom40_welfare_share              0
top20_avg_welfare                   0
top20_welfare_share                 0
top10_avg_welfare                   0
top10_welfare_share                 0
p10_avg_welfare                     0
p90_avg_welfare                     0
top20_bottom20_ratio                0
p90_p10_ratio                       0
welfare_gap                         0
time_code                           0
country_name                        0
gdp_per_capita                      0


,country_code,year,mean_avg_welfare,median_quantile,welfare_std,welfare_iqr,total_pop,is_partial_year,bottom20_avg_welfare,bottom20_welfare_share,...,log_gdp_per_capita,date,decade,inequality_category,gdp_per_capita_norm,life_expectancy_norm,school_enrollment_secondary_norm,electricity_access_norm,development_score,development_tier
0,ALB,2019,14.631818,12.356349,8.727751,9.127375,28472.751151,False,6.225324,0.085093,...,8.711186,2019-01-01,2010,Low inequality,0.043156,0.884985,0.576078,1.0,0.626055,Medium development
1,ALB,2020,14.847158,12.838601,8.448225,9.211061,28031.916239,False,6.267664,0.084429,...,8.704322,2020-01-01,2020,Low inequality,0.042848,0.847881,0.553376,1.0,0.611026,Medium development
2,ARM,2019,9.087722,7.717232,6.495384,4.861010,28849.577912,False,4.060171,0.089355,...,8.433426,2019-01-01,2010,Low inequality,0.032228,0.811701,0.486401,1.0,0.582583,Low development
3,ARM,2020,8.589780,7.608539,4.513978,4.245442,27146.378537,False,4.392951,0.102283,...,8.359294,2020-01-01,2020,Low inequality,0.029789,0.747422,0.498776,1.0,0.568997,Low development
4,ARM,2021,8.941558,7.749302,5.566521,4.569458,26523.512035,False,4.210894,0.094187,...,8.452373,2021-01-01,2020,Low inequality,0.032881,0.722636,0.520681,1.0,0.569049,Low development


## 13. Final Validation of the Analytical Dataset

This final validation checks whether the main analytical columns are fully cleaned
and ready for visualization and insight generation.

In [133]:
important_columns = [
    "country_code",
    "year",
    "mean_avg_welfare",
    "welfare_std",
    "welfare_iqr",
    "bottom20_avg_welfare",
    "top20_avg_welfare",
    "top20_bottom20_ratio",
    "p90_p10_ratio",
    "welfare_gap",
    "total_pop",
    "gdp_per_capita",
    "life_expectancy",
    "school_enrollment_secondary",
    "population_total",
    "electricity_access",
    "unemployment_total",
    "log_gdp_per_capita",
    "development_score"
]

print("Final missing values in important analytical columns:")
display(merged_df[important_columns].isnull().sum().to_frame("Missing Count"))

print("\nDuplicate country-year pairs:")
print(merged_df[["country_code", "year"]].duplicated().sum())

Final missing values in important analytical columns:


,Missing Count
country_code,0
year,0
mean_avg_welfare,0
welfare_std,0
welfare_iqr,0
bottom20_avg_welfare,0
top20_avg_welfare,0
top20_bottom20_ratio,0
p90_p10_ratio,0
welfare_gap,0



Duplicate country-year pairs:
0


## 14. Data Reshaping and Analytical Summaries

This section creates summary tables required for later analysis and visualization.

In [134]:
# Year-level summary
yearly_summary = merged_df.groupby("year").agg(
    mean_welfare=("mean_avg_welfare", "mean"),
    mean_inequality_ratio=("top20_bottom20_ratio", "mean"),
    mean_p90_p10_ratio=("p90_p10_ratio", "mean"),
    mean_development_score=("development_score", "mean"),
    record_count=("country_code", "count")
).reset_index()

print("Yearly Summary:")
display(yearly_summary)

Yearly Summary:


,year,mean_welfare,mean_inequality_ratio,mean_p90_p10_ratio,mean_development_score,record_count
0,2019,33.958047,6.568389,11.562388,0.625733,77
1,2020,38.012446,6.775859,13.205511,0.639108,68
2,2021,32.647042,6.640914,11.881814,0.587976,78
3,2022,31.849304,6.581433,11.607007,0.614588,68
4,2023,36.206624,6.756171,12.082762,0.647676,52
5,2024,18.838444,9.003548,16.480991,0.591833,12


In [135]:
# Country-level summary
country_summary = merged_df.groupby("country_name").agg(
    avg_welfare=("mean_avg_welfare", "mean"),
    avg_inequality_ratio=("top20_bottom20_ratio", "mean"),
    avg_p90_p10_ratio=("p90_p10_ratio", "mean"),
    avg_development_score=("development_score", "mean")
).reset_index()

print("Country Summary:")
display(country_summary.head(10))

Country Summary:


,country_name,avg_welfare,avg_inequality_ratio,avg_p90_p10_ratio,avg_development_score
0,Albania,14.739488,4.535132,6.678129,0.618541
1,Armenia,8.866456,3.957371,5.837771,0.581722
2,Australia,72.711222,5.727196,10.077481,0.803107
3,Austria,80.081438,4.940473,8.460144,0.727342
4,Bangladesh,7.510242,4.555745,6.838854,0.529998
5,Belarus,20.476519,3.455734,4.869509,0.594472
6,Belgium,69.142658,3.910783,5.966378,0.809542
7,Benin,5.266838,5.552207,8.739286,0.244929
8,Bhutan,18.073696,4.237835,6.249656,0.567166
9,Bolivia,22.209334,9.370241,18.457914,0.522413


In [136]:
# Pivot table for inequality by country and year
inequality_pivot = merged_df.pivot_table(
    values="top20_bottom20_ratio",
    index="country_name",
    columns="year"
)

print("Inequality Pivot Table:")
display(inequality_pivot.head(10))

Inequality Pivot Table:


year,2019,2020,2021,2022,2023,2024
country_name,,,,,,
Albania,4.566613,4.503650,NaN,NaN,NaN,NaN
Armenia,4.388254,3.473481,4.001912,4.051212,3.871995,NaN
Australia,NaN,5.727196,NaN,NaN,NaN,NaN
Austria,4.840162,4.777155,5.017282,5.013493,5.054272,NaN
Bangladesh,NaN,NaN,NaN,4.555745,NaN,NaN
Belarus,3.537391,3.374076,NaN,NaN,NaN,NaN
Belgium,4.048912,3.786508,3.916802,3.830618,3.971077,NaN
Benin,NaN,NaN,5.552207,NaN,NaN,NaN
Bhutan,NaN,NaN,NaN,4.237835,NaN,NaN


In [137]:
pip_missing_summary = pd.DataFrame({
    "Column": pip_clean.columns,
    "Missing After Cleaning": pip_clean.isnull().sum().values
}).sort_values("Missing After Cleaning", ascending=False)

wdi_missing_summary = pd.DataFrame({
    "Column": wdi_clean.columns,
    "Missing After Cleaning": wdi_clean.isnull().sum().values
}).sort_values("Missing After Cleaning", ascending=False)

print("PIP Missing Values After Cleaning:")
display(pip_missing_summary)

print("WDI Missing Values After Cleaning:")
display(wdi_missing_summary)

PIP Missing Values After Cleaning:


,Column,Missing After Cleaning
0,country_code,0
1,year,0
2,reporting_level,0
3,welfare_type,0
4,percentile,0
5,avg_welfare,0
6,pop_share,0
7,welfare_share,0
8,quantile,0
9,pop,0


WDI Missing Values After Cleaning:


,Column,Missing After Cleaning
0,year,0
1,time_code,0
2,country_name,0
3,country_code,0
4,gdp_per_capita,0
5,life_expectancy,0
6,school_enrollment_secondary,0
7,population_total,0
8,electricity_access,0
9,unemployment_total,0


## 15. Export the Final Engineered Dataset

In [138]:
merged_df.to_csv("final_poverty_wdi_engineered.csv", index=False)
print("Final cleaned and engineered dataset saved successfully.")

Final cleaned and engineered dataset saved successfully.


## 15. Reflections, Assumptions, and Data Engineering Decisions

To ensure analytical rigor and accurate visual outcomes later, several critical design decisions and trade-offs were made. 

### Data Engineering Assumptions:
1. **Granularity Aggregation:** The raw PIP dataset was originally detailed at the 100-percentile bucket level. It was aggregated to a single country-year level using custom `qcut` logic to generate comparative 20/80 (bottom/top) dynamics. This reduced unnecessary noise but sacrifices micro-level individual variations.
2. **Geographical Scope Constraint:** Only national-level observations were mapped heavily for structural consistency. Rural/Urban-only slices were excluded to avoid comparing partial demographic geometries to national WDI macroeconomic measurements.
3. **Welfare Defintions:** Both **income** and **consumption** metrics were retained interchangeably under the 'welfare' envelope since both are established proxies for developmental modeling, albeit structurally distinct.
4. **Time Window (2019–2024):** Limited deliberately to focus distinctly on contemporary disparities (factoring the COVID-impact and recovery era). 
5. **Missing Value Methodologies:** A *Two-Tier Median Imputation Strategy* was implemented for handling sparse indicators. First attempting a Country-Year localized median fill to respect macro conditions of the individual state, and only deferring to a global mathematical median if that local footprint entirely vanished. Median operations are deliberately favored over 'Mean' fills owing to the extreme right-skew common in socio-economic data (Wealth gap outliers distort standard averages drastically).
6. **Robust Computation Models:** When dividing wealth gaps to trace out ratios (e.g. `top20 / bottom20 ratio`), a `safe_divide()` logic explicitly masks undefined Zero-Divide violations directly into handled `NaNs`.
7. **Temporal Edge Cases:** 2024 metric readings were specifically flagged (`is_partial_year`) acknowledging that multi-lateral agencies (e.g. World Bank) might be operating on preliminary non-finalized metrics for current-year aggregates. 
8. **Development Multi-Dimensional Index Scaling:** Since measuring WDI indices comprises distinct scalar fields (Years for Life Expectancy, USD for GDP per Capita, % for Enrollment), all indices were cast onto a **[0, 1] Min-Max Scaled coordinate system** before generating the unweighted proxy `development_score`. This equalizes vectors safely.

### Potential Biases & Limitations:
- The proxy assumption that **National metrics apply uniformly**. The `development_score` applies universally over the geography despite rural areas invariably demonstrating lower performance metrics inherently.
- Relying entirely upon purely formal and statistically recognized figures, likely rendering gray market, undocumented, or unbanked economies totally invisible herein.